# Semana 5 · Experimento del estudiante: mi primer RAG

Construyes el pipeline completo sobre el corpus de Facturio y, al final, clasificas fallas. Antes de cada bloque hay una predicción: escríbela antes de ejecutar y consérvala aunque falle. Las celdas `TODO` tienen placeholders para que el notebook corra de principio a fin.

Sin `OPENAI_API_KEY` el modelo es local y las respuestas son ilustrativas.

## 0. Preparación

In [ ]:
# En Colab basta ejecutar esta celda una vez. Las semanas 5 a 7 usan las mismas librerías.
!pip install -q sentence-transformers faiss-cpu openai transformers torch

In [ ]:
# Llave de OpenAI. En Colab: panel de secretos (icono de llave) con el nombre OPENAI_API_KEY
# y acceso activado para este notebook. En Jupyter local: variable de entorno antes de abrir.
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass  # fuera de Colab, o sin secreto: se usa la variable de entorno si existe
print("llave cargada" if os.environ.get("OPENAI_API_KEY") else "sin llave: se usará el modelo local")

In [ ]:
import os, re, json, time, warnings, urllib.request
from pathlib import Path
import numpy as np
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# --- Corpus de Facturio: copia local del repositorio si existe; si no, se descarga de GitHub.
CORPUS_ARCHIVOS = [
    "01-manual-facturacion-cfdi.md", "02-manual-usuarios-e-invitaciones.md",
    "03-manual-reportes-y-exportacion.md", "04-manual-integraciones-y-api.md",
    "05-preguntas-frecuentes.md", "06-politica-reembolsos-y-cargos.md",
    "07-politica-cancelacion-y-eliminacion.md", "08-terminos-del-servicio-y-sla.md",
    "09-catalogo-codigos-de-error.md", "10-notas-de-version.md",
    "11-guia-de-seguridad.md", "12-comunicado-externo.md",
]
GOLD_ARCHIVO = "gold_set.jsonl"
RAW = "https://raw.githubusercontent.com/JunBarrs24/itm-maestria-llms/main/shared/datasets/corpus/"

def cargar_corpus(cache="corpus"):
    candidatos = [Path("../../shared/datasets/corpus"), Path("shared/datasets/corpus"), Path(cache)]
    base = next((b for b in candidatos if (b / CORPUS_ARCHIVOS[0]).exists()), None)
    if base is None:
        base = Path(cache); base.mkdir(exist_ok=True)
        for nombre in CORPUS_ARCHIVOS + [GOLD_ARCHIVO]:
            if not (base / nombre).exists():
                urllib.request.urlretrieve(RAW + nombre, base / nombre)
    docs = {n: (base / n).read_text(encoding="utf-8") for n in CORPUS_ARCHIVOS}
    gold = [json.loads(l) for l in (base / GOLD_ARCHIVO).read_text(encoding="utf-8").splitlines() if l.strip()]
    return docs, gold

docs, gold = cargar_corpus()
print(f"{len(docs)} documentos, {sum(len(t.split()) for t in docs.values())} palabras, {len(gold)} preguntas en el gold set")

In [ ]:
# --- Cliente mínimo. Copia de course/shared/llm.py: OpenAI Responses API si hay llave, Qwen local si no.
class LLM:
    def __init__(self, model_local="Qwen/Qwen2.5-1.5B-Instruct", model_api="gpt-5-mini"):
        self.backend = "openai" if os.environ.get("OPENAI_API_KEY") else "local"
        if self.backend == "openai":
            from openai import OpenAI
            self.client, self.model = OpenAI(), model_api
        else:
            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer
            self.tok = AutoTokenizer.from_pretrained(model_local)
            self.net = AutoModelForCausalLM.from_pretrained(model_local, dtype=torch.float32).eval()
            self.model = model_local

    def chat(self, user, system, max_output_tokens=200):
        t = time.perf_counter()
        if self.backend == "openai":
            params = dict(model=self.model, instructions=system, input=user, max_output_tokens=max_output_tokens)
            # Reasoning models (gpt-5*, o*): gastan max_output_tokens en razonar; con "minimal" la respuesta
            # cabe en presupuestos pequeños. Ejercicio: probar "low", "medium", "high" y medir tokens y latencia.
            if self.model.startswith(("gpt-5", "o1", "o3", "o4")):
                params["reasoning"] = {"effort": "minimal"}
            r = self.client.responses.create(**params)
            return {"texto": r.output_text, "in": r.usage.input_tokens, "out": r.usage.output_tokens, "s": time.perf_counter() - t}
        import torch
        msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        texto = self.tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        ids = self.tok(texto, return_tensors="pt").input_ids
        with torch.no_grad():
            out = self.net.generate(ids, max_new_tokens=max_output_tokens, do_sample=False, pad_token_id=self.tok.eos_token_id)
        nuevos = out[0, ids.shape[1]:]
        return {"texto": self.tok.decode(nuevos, skip_special_tokens=True).strip(), "in": int(ids.shape[1]), "out": int(nuevos.shape[0]), "s": time.perf_counter() - t}

llm = LLM()
print("backend:", llm.backend, "|", llm.model)
if llm.backend == "local":
    print("Sin OPENAI_API_KEY: las respuestas de este notebook son ilustrativas (modelo local pequeño).")

## 1. Sin RAG

**Predicción.** Sin documentación, ¿qué número dará el modelo para el plazo de reembolso? ¿Y para la vigencia del código 2FA?

In [ ]:
# TODO: escribe tus predicciones antes de ejecutar la siguiente celda.
MI_PREDICCION_REEMBOLSO = None   # por ejemplo "30 días"
MI_PREDICCION_2FA = None         # por ejemplo "30 segundos"

In [ ]:
SYS_SIN_RAG = "Eres el asistente de soporte de Facturio, un SaaS mexicano de facturación. Responde en una oración."
preguntas_trampa = [
    "¿Cuántos días tengo para solicitar un reembolso de un cargo en Facturio?",
    "¿Cuánto tiempo es válido un código de autenticación de dos factores en Facturio?",
]
for q in preguntas_trampa:
    r = llm.chat(q, system=SYS_SIN_RAG, max_output_tokens=60)
    print(f"{q}\n   -> {r['texto']}\n")

In [ ]:
# TODO: responde en texto. ¿El modelo inventó, generalizó o dijo que no sabía? ¿Cuál de las tres conductas es peor para una mesa de soporte y por qué?
RESPUESTA_1 = """
"""

## 2. Parse y chunk

**Predicción.** El corpus tiene 12 documentos y unas 10,500 palabras. ¿Cuántos chunks saldrán con ventanas de 800 caracteres? ¿Y con un chunk por encabezado?

In [ ]:
MI_PREDICCION_CHUNKS_FIJO = None
MI_PREDICCION_CHUNKS_SECCION = None

In [ ]:
def chunk_fijo(archivo, texto, tam=800, overlap=100):
    """Ventanas de `tam` caracteres con traslape. Guarda el último encabezado visto como sección."""
    chunks, seccion, pos = [], "", 0
    while pos < len(texto):
        enc = re.findall(r"^#{2,3}\s+(.*)$", texto[:pos + tam], re.M)
        if enc:
            seccion = enc[-1].strip()
        chunks.append({"archivo": archivo, "seccion": seccion, "texto": texto[pos:pos + tam]})
        pos += tam - overlap
    return chunks

def chunk_por_seccion(archivo, texto):
    """Un chunk por encabezado ## o ###: respeta la estructura que el autor ya puso."""
    chunks, seccion, buf = [], None, []
    for linea in texto.splitlines():
        m = re.match(r"^(#{2,3})\s+(.*)$", linea)
        if m:
            if buf and seccion:
                chunks.append({"archivo": archivo, "seccion": seccion, "texto": "\n".join(buf).strip()})
            seccion, buf = m.group(2).strip(), [linea]
        else:
            buf.append(linea)
    if buf and seccion:
        chunks.append({"archivo": archivo, "seccion": seccion, "texto": "\n".join(buf).strip()})
    return chunks

fijo = [c for n, t in docs.items() for c in chunk_fijo(n, t)]
seccion = [c for n, t in docs.items() for c in chunk_por_seccion(n, t)]
for nombre, cs in [("fijo 800/100", fijo), ("por sección", seccion)]:
    tam = [len(c["texto"]) for c in cs]
    print(f"{nombre:14s} {len(cs):4d} chunks   chars: media {int(np.mean(tam))}, min {min(tam)}, max {max(tam)}")
print("\nUn chunk por sección, por ejemplo:")
ej = next(c for c in seccion if c["archivo"].startswith("06") and c["seccion"] == "Plazo")
print(ej["archivo"], "#", ej["seccion"], "\n", ej["texto"][:400])

## 3. Embed e indexar

In [ ]:
from sentence_transformers import SentenceTransformer
emb = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")   # un encoder (semana 2)
chunks = seccion                                       # la unidad de retrieval de esta sesión
t = time.perf_counter()
E = emb.encode([c["texto"] for c in chunks], normalize_embeddings=True, batch_size=32)
print("matriz de embeddings:", E.shape, f"en {time.perf_counter()-t:.1f} s   (una fila por chunk, normalizada)")

pregunta = "¿Cuántos días tengo para solicitar el reembolso de un cargo?"
v = emb.encode([pregunta], normalize_embeddings=True)[0]
scores = E @ v                                          # producto punto = coseno, porque ambos están normalizados
orden = np.argsort(-scores)[:5]
print("\nTop-5 por coseno (índice numpy en cinco líneas):")
for i in orden:
    print(f"  {scores[i]:.3f}  {chunks[i]['archivo']} # {chunks[i]['seccion']}")

In [ ]:
import faiss
indice = faiss.IndexFlatIP(E.shape[1])      # producto interno sobre vectores normalizados = coseno
indice.add(E.astype(np.float32))
D, I = indice.search(v.astype(np.float32).reshape(1, -1), 5)
print("FAISS devuelve los mismos cinco:", [int(i) for i in I[0]] == [int(i) for i in orden])

def recuperar(pregunta, k=5):
    q = emb.encode([pregunta], normalize_embeddings=True).astype(np.float32).reshape(1, -1)
    D, I = indice.search(q, k)
    return [{**chunks[i], "score": float(d)} for d, i in zip(D[0], I[0])]

**Predicción.** De estas tres preguntas, ¿cuál crees que recuperará peor evidencia y por qué?

1. ¿Cuánto dura el enlace de una invitación?
2. ¿Qué cabecera HTTP usa la API v2?
3. ¿Qué pasa si me equivoco?

In [ ]:
MI_PREDICCION_PEOR_RETRIEVAL = None   # 1, 2 o 3, con una línea de por qué
for q in ["¿Cuánto dura el enlace de una invitación?", "¿Qué cabecera HTTP usa la API v2?", "¿Qué pasa si me equivoco?"]:
    print(q)
    for c in recuperar(q, 3):
        print(f"   {c['score']:.3f}  {c['archivo']} # {c['seccion']}")

## 4. Contexto y generación

In [ ]:
SYSTEM = (
    "Eres el asistente de soporte de Facturio. Respondes únicamente con la evidencia que se te entrega, "
    "en español y en máximo tres oraciones. Al final de cada afirmación cita la fuente entre corchetes "
    "con el formato [archivo#sección]. Si la evidencia no contiene la respuesta, responde exactamente: "
    "No tengo evidencia en la documentación para responder eso."
)

def construir_contexto(chunks_rec):
    partes = []
    for i, c in enumerate(chunks_rec, start=1):
        partes.append(f'<evidencia id="{i}" fuente="{c["archivo"]}#{c["seccion"]}">\n{c["texto"]}\n</evidencia>')
    return "\n\n".join(partes)

def construir_prompt(pregunta, chunks_rec):
    return f"{construir_contexto(chunks_rec)}\n\n<pregunta>\n{pregunta}\n</pregunta>"

rec = recuperar(pregunta, k=3)
print(construir_prompt(pregunta, rec)[:1500], "\n...")

In [ ]:
log = []   # registro por llamada: tokens, latencia, fuentes

def responder(pregunta, k=5, system=SYSTEM):
    rec = recuperar(pregunta, k)
    r = llm.chat(construir_prompt(pregunta, rec), system=system)
    fuentes = [f"{c['archivo']}#{c['seccion']}" for c in rec]
    log.append({"pregunta": pregunta, "in": r["in"], "out": r["out"], "s": r["s"], "fuentes": fuentes})
    return r["texto"], fuentes

def mostrar(pregunta, k=5):
    texto, fuentes = responder(pregunta, k)
    print(f"Pregunta: {pregunta}\nRespuesta: {texto}\nFuentes top-{k}:")
    for f in fuentes:
        print("   ", f)
    print()

mostrar("¿Cuántos días tengo para solicitar el reembolso de un cargo?")

## 5. Answerability

**Predicción.** Si quitas del system prompt la frase de "No tengo evidencia", ¿qué hará el modelo con la pregunta de Guadalajara?

In [ ]:
MI_PREDICCION_SIN_FRASE = None
mostrar("¿Facturio tiene oficina en Guadalajara?")
SYSTEM_SIN_ANSWERABILITY = SYSTEM.split(" Si la evidencia")[0]
texto, fuentes = responder("¿Facturio tiene oficina en Guadalajara?", system=SYSTEM_SIN_ANSWERABILITY)
print("Sin la instrucción de answerability:", texto)

## 6. Cinco preguntas tuyas y su diagnóstico

Escribe cinco preguntas sobre Facturio (al menos una cuya respuesta no esté en la documentación). Para cada una, ejecuta, mira las fuentes recuperadas y clasifica: `correcta`, `falla_retrieval` (la evidencia no llegó), `falla_generacion` (la evidencia llegó y el modelo la ignoró o deformó), `sin_respuesta_correcta` (no había evidencia y lo dijo).

In [ ]:
# TODO: cinco preguntas propias. Cambia los ejemplos.
MIS_PREGUNTAS = [
    "¿Cuántos administradores puede tener una cuenta Pro?",
    "¿Qué hago si el correo de invitación no llega?",
    "¿Cuánto cuesta el plan Empresa?",
    "¿Puedo pagar en efectivo en una tienda?",
    "¿Qué significa el error E-2001?",
]
for q in MIS_PREGUNTAS:
    mostrar(q)

In [ ]:
# TODO: un diagnóstico por pregunta, con una línea de justificación que mencione la fuente recuperada.
MI_DIAGNOSTICO = {
    MIS_PREGUNTAS[0]: ("", ""),   # (clasificación, justificación)
    MIS_PREGUNTAS[1]: ("", ""),
    MIS_PREGUNTAS[2]: ("", ""),
    MIS_PREGUNTAS[3]: ("", ""),
    MIS_PREGUNTAS[4]: ("", ""),
}
for q, (clase, just) in MI_DIAGNOSTICO.items():
    print(f"{clase or '(pendiente)':22s} {q}")

## 7. La perilla k

**Predicción.** Con k = 1 y con k = 10, ¿cambia la respuesta a la pregunta del reembolso? ¿Qué pasa con los tokens de entrada?

In [ ]:
MI_PREDICCION_K = None
for k in (1, 10):
    texto, fuentes = responder("¿Cuántos días tengo para solicitar el reembolso de un cargo?", k=k)
    print(f"k={k:2d}  tokens in={log[-1]['in']:5d}  ->  {texto[:120]}")

## 8. Costo

In [ ]:
tot_in = sum(l["in"] for l in log); tot_out = sum(l["out"] for l in log)
print(f"llamadas: {len(log)}   tokens in: {tot_in}   tokens out: {tot_out}   latencia media: {np.mean([l['s'] for l in log]):.1f} s")
print(f"tokens de contexto por pregunta (media): {tot_in/len(log):.0f}   ->  el retrieval decide cuánto pagas por llamada")

## Cierre

Responde en dos o tres líneas: de las fallas que viste, ¿cuántas se arreglan cambiando el prompt y cuántas requieren cambiar el retrieval? Esa es la pregunta de la semana 6.

In [ ]:
RESPUESTA_CIERRE = """
"""